## Check initial WH events

Checks:
 1. The setup event should be one event rather than two.
 2. The first actual setup event should be either `setup_right_sym` (`trigger` 3) or
`setup_left_sym` (`trigger` 2).  It should be the first event.
 3. The setup event should have sample number 1 and onset 1/srate.  (The srate = 1100).
 4. The initial face event should be of type `initial_face_event`. It should be the second event in the file.


In [ ]:
from hed.tools.io_utils import get_file_list, make_file_dict
from hed.tools.data_utils import get_new_dataframe

bids_root_path = 'G:/WH_working2'
bids_files = get_file_list(bids_root_path, extensions=[".tsv"], name_suffix="_events")
file_dict = make_file_dict(bids_files, indices=(0, -2))
bids_skip = ['onset', 'duration', 'sample', 'response_time', 'stim_file', 'HED']
srate = 1100

print(f"\nBIDS form of the events: {len(file_dict)}")
status = {}
for key, file in file_dict.items():
    df = get_new_dataframe(file)
    status[key] = []
    # Handle the setup event
    if df.loc[df.index[0], 'event_type'] == 'setup':
        df.drop([0], inplace=True)
        status[key].append('Dropped initial setup event')
    if df.loc[df.index[0], 'event_type'] == 'right_sym':
        df.loc[df.index[0], 'event_type'] = 'setup_right_sym'
        df.loc[df.index[0], 'trigger'] = 3
        status[key].append('Replaced right_sym with setup_right_sym and trigger 3')
    elif df.loc[df.index[0], 'event_type'] == 'left_sym':
        df.loc[df.index[0], 'event_type'] = 'setup_left_sym'
        df.loc[df.index[0], 'trigger'] = 2
        status[key].append('Replaced left_sym with setup_left_sym and trigger 2')
    else:
        print(f"ERROR: bad first line in {file}")
        status[key].append('ERROR: No initial setup event')
        continue
    df.loc[df.index[0], 'sample'] = 1.0
    df.loc[df.index[0], 'onset'] = 1.0/srate

    # Now handle the initial face events
    if df.loc[df.index[1], 'event_type'] == 'show_face':
        df.loc[df.index[1], 'event_type'] = 'show_face_initial'
        df.loc[df.index[1], 'trigger'] = int(df.loc[df.index[1], 'trigger']) + 20
    else:
        print(f"ERROR: second event in {file} should be a show_face replaced by show_face_initial")
        status[key].append('ERROR: second event in should be a show_face')
        continue

    # filename = file[:-4] + "_temp1.tsv"
    # df.to_csv(filename, sep='\t', index=False)
    break


for key, status_item in status.items():
    print(f"{key}")
    if status_item:
        for msg in status_item:
            print(f"\t{msg}")